# Pillar 3I — controlled learning-rate tune of the 3b recipe on V14

**One variable changes vs the failed 3g: the learning rate.** Everything else is the *exact* 3b recipe (`train_path_b`, `--target-temperature 0.5`, `--batch-size 32768`, `--warmup-epochs 1`, `--amp --compile`, color+dihedral aug), the *same* V14 tensor 3g used (`v14_pillar3f.pt`), and the *same* warm-start (`pillar3f`).

**Why:** 3g (this recipe at **lr 3e-4**) degraded pillar3f from **epoch 1**, uniformly across all percentiles (best-val ckpt worse than the warm-start). 3b used the same lr 3e-4 on a *much weaker* policy (pillar3a) and gained +15%. Hypothesis to test: on the now ~4-5× stronger pillar3f, lr 3e-4 is too aggressive — a **gentler lr** may let the distillation improve instead of overshoot. This notebook isolates exactly that.

**Sweep plan (one lr per run — Colab can't fit several × 20 epochs):** start `LR = 1e-4`; if it still degrades, drop to `5e-5`, then `3e-5`. Change `LR` and `RUN` in the config cell and re-run.

**Eval by GAMEPLAY, not val loss.** Val loss is a weak predictor here (3b gained +15% from a ~0.004 val drop). All epoch checkpoints are saved to Drive; eval each with `eval_policy` on a seed range and read the **floor (P1–P25, <1000)** — never per-seed. Bar to beat (pillar3f, 1000 seeds 779000-779999, uncapped): **mean 33837, P1 968, P5 2457, P10 4507, P25 11142, <1000 1.1%**.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_pillar3d_v2.tar.gz` — current code (has `train_path_b.py` + `gumbel.py` + `dataset.py`).
2. `v14_pillar3f.pt.gz` — the V14 tensor (gzip of the 5.75 GB file; the one 3g trained on).
3. `pillar3f.pt` — warm-start weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3f.pt', '/content/alphatrain/data/pillar3f.pt')
print(f'Warm-start (pillar3f): {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')

# Copy gzipped tensor to local disk FIRST, verify integrity, THEN decompress.
# (Streaming through Drive FUSE can silently truncate files this large.)
t0 = time.time()
!cp {DRIVE}/v14_pillar3f.pt.gz /content/v14_pillar3f.pt.gz
print(f'.gz copied: {os.path.getsize("/content/v14_pillar3f.pt.gz"):,} bytes')
!gunzip -t /content/v14_pillar3f.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v14_pillar3f.pt.gz > /content/alphatrain/data/v14_pillar3f.pt
pt_size = os.path.getsize('/content/alphatrain/data/v14_pillar3f.pt')
EXPECTED_PT = 5_748_249_183
assert pt_size == EXPECTED_PT, (
    f'Decompressed .pt size wrong! got {pt_size} expected {EXPECTED_PT}. '
    f'Re-upload v14_pillar3f.pt.gz to Drive.')
print(f'V14 tensor: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
!rm /content/v14_pillar3f.pt.gz

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG — the ONLY thing to change between sweep points =====
LR     = 1e-4     # 3g used 3e-4 (failed). Sweep: 1e-4 -> 5e-5 -> 3e-5.
RUN    = 'pillar3i_lr1e-4'   # <- rename to match LR each run (keeps Drive ckpts distinct)
EPOCHS = 20       # save every epoch; pick the best by GAMEPLAY floor, not val
# Everything below is the frozen 3b recipe.
print(f'RUN={RUN}  LR={LR}  EPOCHS={EPOCHS}')

In [ ]:
# Pillar 3I = the 3b recipe on V14, ONLY --lr changed (vs 3g's 3e-4).
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v14_pillar3f.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --epochs {EPOCHS} --batch-size 32768 --lr {LR} --warmup-epochs 1 \
    --target-temperature 0.5 \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN}
# If val is still falling at epoch 20, extend with --resume (lower lr learns slower).

In [ ]:
# Copy every epoch checkpoint to Drive (val is a weak predictor -> eval gameplay per-epoch).
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst); print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}'); print(f'Saved {DRIVE}/{RUN}_{f}')

## Eval (do this for each epoch checkpoint — gameplay floor, not val)
Pull the epoch checkpoints back and run `eval_policy` on a fixed seed range, distribution-level:
```
python -m scripts.eval_policy --model <ckpt> --device cuda --batch 1024 \
    --seed-start 779000 --seed-end 779999
```
Compare the **floor (P1, P5, P10, P25, <1000)** against pillar3f: **mean 33837, P1 968, P5 2457, P10 4507, P25 11142, <1000 1.1%**.
- Any epoch with floor **≥ pillar3f** → the gentler lr recovered improvement; pick the best.
- All epochs **< pillar3f** (like 3g) → lr 1e-4 isn't enough; drop to 5e-5, then 3e-5.
- If even 3e-5 only ever *degrades* → lr is not the lever; the recipe needs a different change (back to the reviewers).